# Calibration stack

For the given nights (`N_START` .. `N_STOP`, by convention blocks of 10 nights) of one telescope: finds the zeros and the
darks, checks their headers, and makes per-pixel maps, first of the zeros, then of the darks with the bias
subtracted. Flats may come here later.

- **Bias** (`~/data/ocm_bias/<tel>/<tel>_<N_START>_<N_STOP>_<map>.fits`): `mean`, `std` (ADU) and `nclip`.
- **Dark** (`~/data/ocm_dark/<tel>/<tel>_<N_START>_<N_STOP>_<map>.fits`): `mean`, `std_excess` (ADU/s)
  and `nclip`. From every dark the bias `mean` map of the same nights is subtracted and the result divided by the
  dark's own `EXPTIME`.

`mean` and `std` are computed without the values rejected by clipping (cosmics); `nclip` is how many values of the pixel
were rejected. Frames are read from the OCM archive `/data/fits/<tel>/raw/<night>/` and not copied. The maps are shown
by `bias_view.ipynb` and `dark_view.ipynb`, their trends in time by `calib_trends.ipynb`.

In [68]:
import os
import glob
import numpy as np
from astropy.io import fits
from tqdm.notebook import tqdm

TEL = 'zb08'
N_START = 1310     # first (oldest) night, int(JD) - 2460000
N_STOP = 1320      # last (newest) night

# less important
T_MAX = -55
MIN_EXPTIME = 600  # [s] shortest exposure time of the darks to stack
MIN_NDARK = 4      # fewest darks of one exposure time to stack
SEG = 256          # height of one band in rows
SIGMA = 5          # clipping limit, in units of the pixel noise

RAW_DIR = f'/data/fits/{TEL}/raw'
BIAS_DIR = os.path.expanduser(f'~/data/ocm_bias/{TEL}')
DARK_DIR = os.path.expanduser(f'~/data/ocm_dark/{TEL}')

## Find the frames

One pass over the headers of all frames of the nights (the slow part, ~0.17 s per file over NFS), collecting the zeros
and the darks.

In [ ]:
# old: reading every header (replaced by the frames index in the next cell)
# zeros = []  # (file, T-CAM)
# darks = []  # (file, EXPTIME, T-CAM)
#
# for night in tqdm(range(N_START, N_STOP + 1), desc='nights'):
#     files = sorted(glob.glob(f'{RAW_DIR}/{night}/*.fits'))
#     for f in tqdm(files, desc=f'night {night}', leave=False):
#         h = fits.getheader(f)
#         if h['IMAGETYP'] == 'zero':
#             zeros.append((f, h['T-CAM']))
#         if h['IMAGETYP'] == 'dark':
#             darks.append((f, h['EXPTIME'], h['T-CAM']))
#
# print(len(zeros), 'zeros', len(darks), 'darks')

In [ ]:
# the frames of these nights from the index made by frames_index.ipynb
import pandas as pd
FRAMES_DIR = os.path.expanduser(f'~/data/ocm_frames/{TEL}')
nights = [n for n in range(N_START, N_STOP + 1) if os.path.exists(f'{FRAMES_DIR}/{n}.parquet')]
missing = [n for n in range(N_START, N_STOP + 1) if n not in nights]
if missing:
    print('nights not in the index (no frames, or run frames_index.ipynb):', missing)
frames = pd.concat([pd.read_parquet(f'{FRAMES_DIR}/{n}.parquet') for n in nights], ignore_index=True)

z = frames[frames['IMAGETYP'] == 'zero']
d = frames[frames['IMAGETYP'] == 'dark']
zeros = list(zip(z['file'], z['T-CAM']))                 # (file, T-CAM)
darks = list(zip(d['file'], d['EXPTIME'], d['T-CAM']))   # (file, EXPTIME, T-CAM)

print(len(zeros), 'zeros', len(darks), 'darks')

## Bias: check the headers

Only zeros with the CCD at its set temperature (-60 C).

In [70]:
zero_list = []
for f, tcam in zeros:
    if tcam < T_MAX:
        zero_list.append(f)
    else:
        print('rejected (T-CAM)', f, tcam)

print(len(zero_list), 'zeros used')

155 zeros used


## Bias: stack

All frames together would not fit in memory (100 frames of 2048 x 2048 in float32 = 1.6 GB, more for bigger cameras),
so the image is cut into horizontal bands of `SEG` rows. For one band the same rows are read from every zero, which
gives a small cube `(number of zeros, SEG, 2048)`; the statistics are computed along the first axis (over zeros),
i.e. for every pixel separately, and written into the matching rows of the full maps.

**Clipping.** A value is rejected if it is further from the pixel median than `SIGMA` x the pixel noise. The pixel noise
is its MAD x 1.4826 (= std for Gaussian noise; a cosmic does not change it, unlike the std), but not less than the
read noise: with few frames the MAD of one pixel can come out much too small by chance (e.g. 0.7 ADU instead of 7 with
12 frames), and normal values would then be rejected. For the zeros the read noise is the median of the pixel MADs in
the band (one number, ~7 ADU on wk06). One pass, no iterations; at least the values at the median always stay, so
`mean` and `std` are never NaN.

In [71]:
h = fits.getheader(zero_list[0])
ny, nx = h['NAXIS2'], h['NAXIS1']

bias_mean = np.zeros((ny, nx), np.float32)
bias_std = np.zeros((ny, nx), np.float32)
bias_nclip = np.zeros((ny, nx), np.int16)

for y0 in tqdm(range(0, ny, SEG), desc='bias strips'):
    y1 = min(y0 + SEG, ny)

    # rows y0..y1 of every zero; .section reads only this part of the file, not the whole image
    paski = []
    for f in tqdm(zero_list, desc=f'rows {y0}', leave=False):
        with fits.open(f) as hdul:
            paski.append(hdul[0].section[y0:y1, :].astype(np.float32))
    paski = np.array(paski)

    # axis=0 = over the zeros, so every pixel gets its own value
    med = np.median(paski, axis=0)
    mad = 1.4826 * np.median(np.abs(paski - med), axis=0)
    noise = np.maximum(mad, np.median(mad))       # pixel noise, not less than the read noise
    bad = np.abs(paski - med) > SIGMA * noise     # rejected values
    kept = np.where(bad, np.nan, paski)
    bias_mean[y0:y1] = np.nanmean(kept, axis=0)
    bias_std[y0:y1] = np.nanstd(kept, axis=0)
    bias_nclip[y0:y1] = bad.sum(axis=0)

for name, m in [('mean', bias_mean), ('std', bias_std), ('nclip', bias_nclip)]:
    print(f'{name:6s} median {np.median(m):9.3f}   min {m.min():9.3f}   max {m.max():9.3f}')

bias strips:   0%|          | 0/8 [00:00<?, ?it/s]

rows 0:   0%|          | 0/155 [00:00<?, ?it/s]

rows 256:   0%|          | 0/155 [00:00<?, ?it/s]

rows 512:   0%|          | 0/155 [00:00<?, ?it/s]

rows 768:   0%|          | 0/155 [00:00<?, ?it/s]

rows 1024:   0%|          | 0/155 [00:00<?, ?it/s]

rows 1280:   0%|          | 0/155 [00:00<?, ?it/s]

rows 1536:   0%|          | 0/155 [00:00<?, ?it/s]

rows 1792:   0%|          | 0/155 [00:00<?, ?it/s]

mean   median   299.233   min   295.325   max   304.520
std    median     8.380   min     5.422   max    14.410
nclip  median     3.000   min     0.000   max    14.000


## Bias: save

In [72]:
hdr = fits.Header()
hdr['TELESCOP'] = TEL
hdr['NIGHT-BG'] = (N_START, 'First night')
hdr['NIGHT-EN'] = (N_STOP, 'Last night')
hdr['NZERO'] = (len(zero_list), 'Number of zeros in the stack')
hdr['SIGCLIP'] = (SIGMA, 'Clip at SIGMA x max(pixel MAD, read noise)')
for i, f in enumerate(zero_list):
    hdr[f'FILE{i + 1:03d}'] = os.path.basename(f)

# one FITS file per map, e.g. ~/data/ocm_bias/wk06/wk06_1330_1340_mean.fits
os.makedirs(BIAS_DIR, exist_ok=True)
for name, m in [('mean', bias_mean), ('std', bias_std), ('nclip', bias_nclip)]:
    out = f'{BIAS_DIR}/{TEL}_{N_START}_{N_STOP}_{name}.fits'
    fits.writeto(out, m, hdr, overwrite=True)
    print(out)

/home/mgorski/data/ocm_bias/zb08/zb08_1310_1320_mean.fits
/home/mgorski/data/ocm_bias/zb08/zb08_1310_1320_std.fits
/home/mgorski/data/ocm_bias/zb08/zb08_1310_1320_nclip.fits


## Dark: choose the darks

Only darks with the CCD at its set temperature (-60 C). Darks of different exposure times are not mixed (the read
noise in ADU/s differs), so one exposure time is chosen: of those at least `MIN_EXPTIME` s long with at least
`MIN_NDARK` darks, the one with the most darks (with equal numbers the longer one). If there is none, a message is
printed and the dark cells below do nothing (the bias is already saved).

In [73]:
good = []  # (file, exposure time) of the darks with the CCD at its set temperature
for f, t, tcam in darks:
    if tcam < T_MAX:
        good.append((f, t))
    else:
        print('rejected (T-CAM)', f, tcam)

# exposure times that qualify, as (number of darks, exposure time)
candidates = []
for t in sorted(set(round(t) for f, t in good)):
    n = sum(1 for f, e in good if round(e) == t)
    print(f'EXPTIME {t:6d} s: {n} darks')
    if t >= MIN_EXPTIME and n >= MIN_NDARK:
        candidates.append((n, t))
if candidates:
    n, EXPTIME = max(candidates)  # the most darks; with equal numbers the longer exposure
    dark_list = [f for f, t in good if round(t) == EXPTIME]
    dark_exptimes = [t for f, t in good if round(t) == EXPTIME]  # EXPTIME from the header of every dark used
    print(f'used: EXPTIME {EXPTIME} s, {len(dark_list)} darks')
else:
    dark_list = []  # the dark cells below then do nothing
    print(f'no darks to stack: no exposure time with at least {MIN_NDARK} darks of at least {MIN_EXPTIME} s')

no darks to stack: no exposure time with at least 4 darks of at least 600 s


## Dark: stack

The same bands. Every band of every dark: (raw - `bias_mean`) / its EXPTIME from the header, in ADU/s. The clipping as
for the zeros, but the read noise of every pixel is known now: its `bias_std` (divided by EXPTIME, to ADU/s). A hot
pixel scatters more than that, and then its own MAD is used.

`std_excess`: the scatter of a pixel over the darks has two parts, the read noise (`bias_std`) and the noise of the
dark signal itself. In variances they add up, so the dark part is `sqrt(std_dark**2 - bias_std**2)`, both in ADU per
exposure, then divided by EXPTIME. Where the difference is negative it is set to 0. With ~10 darks the std of one pixel
is uncertain by ~20 %, so for a typical pixel `std_excess` is mostly noise; it shows the pixels that scatter clearly
more than the read noise.

In [74]:
if dark_list:  # nothing to do without darks
    dark_mean = np.zeros((ny, nx), np.float32)
    dark_std = np.zeros((ny, nx), np.float32)
    dark_nclip = np.zeros((ny, nx), np.int16)

    for y0 in tqdm(range(0, ny, SEG), desc='dark bands'):
        y1 = min(y0 + SEG, ny)

        # rows y0..y1 of every dark, bias subtracted, in ADU/s
        paski = []
        for f, t in tqdm(zip(dark_list, dark_exptimes), total=len(dark_list), desc=f'rows {y0}', leave=False):
            with fits.open(f) as hdul:
                paski.append((hdul[0].section[y0:y1, :].astype(np.float32) - bias_mean[y0:y1]) / t)
        paski = np.array(paski)

        med = np.median(paski, axis=0)
        mad = 1.4826 * np.median(np.abs(paski - med), axis=0)
        noise = np.maximum(mad, bias_std[y0:y1] / EXPTIME)  # pixel noise, not less than its read noise
        bad = np.abs(paski - med) > SIGMA * noise
        kept = np.where(bad, np.nan, paski)
        dark_mean[y0:y1] = np.nanmean(kept, axis=0)
        dark_std[y0:y1] = np.nanstd(kept, axis=0)
        dark_nclip[y0:y1] = bad.sum(axis=0)

    # dark noise without the read noise; dark_std * EXPTIME = std in ADU per exposure, like bias_std
    excess = (dark_std * EXPTIME) ** 2 - bias_std ** 2
    dark_std_excess = (np.sqrt(np.clip(excess, 0, None)) / EXPTIME).astype(np.float32)

    for name, m in [('mean', dark_mean), ('std_excess', dark_std_excess), ('nclip', dark_nclip)]:
        print(f'{name:10s} median {np.median(m):10.6f}   min {m.min():10.6f}   max {m.max():10.6f}')

## Dark: save

In [75]:
if dark_list:  # nothing to do without darks
    hdr = fits.Header()
    hdr['TELESCOP'] = TEL
    hdr['NIGHT-BG'] = (N_START, 'First night')
    hdr['NIGHT-EN'] = (N_STOP, 'Last night')
    hdr['EXPTIME'] = (np.mean(dark_exptimes), '[s] Exposure time of the darks, from headers')
    hdr['BUNIT'] = 'ADU/s'
    hdr['NDARK'] = (len(dark_list), 'Number of darks in the stack')
    hdr['BIAS'] = (f'{TEL}_{N_START}_{N_STOP}', 'Bias stack subtracted (mean map)')
    hdr['SIGCLIP'] = (SIGMA, 'Clip at SIGMA x max(pixel MAD, read noise)')
    for i, f in enumerate(dark_list):
        hdr[f'FILE{i + 1:03d}'] = os.path.basename(f)

    # one FITS file per map, e.g. ~/data/ocm_dark/wk06/wk06_1330_1340_mean.fits (the exposure time is in the header)
    os.makedirs(DARK_DIR, exist_ok=True)
    # the full std of the darks (read noise included) is not saved, only its dark part std_excess
    for name, m in [('mean', dark_mean), ('std_excess', dark_std_excess), ('nclip', dark_nclip)]:
        out = f'{DARK_DIR}/{TEL}_{N_START}_{N_STOP}_{name}.fits'
        fits.writeto(out, m, hdr, overwrite=True)
        print(out)